In [ ]:
# @title Set ONLINE = False if running locally
ONLINE = True

In [ ]:
#@title Imports
import numpy as np
import pandas as pd
import math
from numba import njit
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import gdown
import os

In [ ]:
# Download training and submission data
if ONLINE:
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)
    url = 'https://drive.google.com/file/d/1KMiakfzTWYr4QJU9S5DC9e5KIo_8bb24/view?usp=sharing'
    output = 'sampleSubmission.csv'
    gdown.download(url, output, fuzzy=True)

In [ ]:
# Load data
data_pd = pd.read_csv('data_train.csv')

def extract_users_items_predictions(data):
    """
    Extract users, items (movies), and predictions from the dataset.
    """
    users, movies = data.Id.str.extract('r(\d+)_c(\d+)').astype(int).values.T - 1
    predictions = data.Prediction.values
    return users, movies, predictions

# Split data
train_size = 0.9
train_pd, test_pd = train_test_split(data_pd, train_size=train_size, random_state=42)
train_users, train_movies, train_predictions = extract_users_items_predictions(train_pd)

number_of_users, number_of_movies = 10000, 1000

# Create full matrix of observed values
data = np.full((number_of_users, number_of_movies), np.mean(train_predictions))
mask = np.zeros((number_of_users, number_of_movies))  # 0 -> unobserved value, 1->observed value

for user, movie, pred in zip(train_users, train_movies, train_predictions):
    data[user, movie] = pred
    mask[user, movie] = 1

def rmse(x, y):
    """
    Calculate Root Mean Squared Error (RMSE).
    """
    return math.sqrt(mean_squared_error(x, y))

test_users, test_movies, test_predictions = extract_users_items_predictions(test_pd)

class BMF:
    def __init__(self, n_users, n_items, n_factors, n_iter, alpha, beta, random_state=None):
        """
        Initialize the BMF model with given parameters.

        Parameters:
        n_users (int): Number of users.
        n_items (int): Number of items (movies).
        n_factors (int): Number of latent factors.
        n_iter (int): Number of iterations.
        alpha (float): Learning rate.
        beta (float): Regularization parameter.
        random_state (int, optional): Seed for random number generator.
        """
        self.n_users = n_users
        self.n_items = n_items
        self.n_factors = n_factors
        self.n_iter = n_iter
        self.alpha = alpha
        self.beta = beta
        self.global_bias = 0  # Global bias term
        self.user_bias = np.zeros(n_users)  # Bias term for each user
        self.item_bias = np.zeros(n_items)  # Bias term for each item

        if random_state is not None:
            np.random.seed(random_state)

        # Latent factors for users and items
        self.user_factors = np.random.normal(scale=1./n_factors, size=(n_users, n_factors))
        self.item_factors = np.random.normal(scale=1./n_factors, size=(n_items, n_factors))

    def fit(self, train_users, train_movies, train_predictions):
        """
        Fit the BMF model to the training data.

        Parameters:
        train_users (array-like): User indices for training data.
        train_movies (array-like): Movie indices for training data.
        train_predictions (array-like): Predictions for training data.
        """
        self.global_bias, self.user_bias, self.item_bias, self.user_factors, self.item_factors = self._fit_jit(
            train_users, train_movies, train_predictions, self.n_iter, self.alpha, self.beta,
            self.global_bias, self.user_bias, self.item_bias, self.user_factors, self.item_factors)

    @staticmethod
    @njit
    def _fit_jit(train_users, train_movies, train_predictions, n_iter, alpha, beta, global_bias, user_bias, item_bias, user_factors, item_factors):
        """
        Perform the training using Numba's JIT compilation for optimization.

        Parameters:
        train_users (array-like): User indices for training data.
        train_movies (array-like): Movie indices for training data.
        train_predictions (array-like): Predictions for training data.
        n_iter (int): Number of iterations.
        alpha (float): Learning rate.
        beta (float): Regularization parameter.
        global_bias (float): Global bias term.
        user_bias (array-like): Bias term for each user.
        item_bias (array-like): Bias term for each item.
        user_factors (array-like): Latent factors for users.
        item_factors (array-like): Latent factors for items.

        Returns:
        Updated global bias, user bias, item bias, user factors, and item factors.
        """
        for iteration in range(n_iter):
            for idx in range(len(train_users)):
                u = train_users[idx]  # User index
                i = train_movies[idx]  # Movie index
                r = train_predictions[idx]  # Actual rating

                # Predicted rating
                prediction = global_bias + user_bias[u] + item_bias[i] + np.dot(user_factors[u], item_factors[i])
                error = r - prediction  # Prediction error

                # Update biases and latent factors
                global_bias += alpha * error
                user_bias[u] += alpha * (error - beta * user_bias[u])
                item_bias[i] += alpha * (error - beta * item_bias[i])
                user_factors[u] += alpha * (error * item_factors[i] - beta * user_factors[u])
                item_factors[i] += alpha * (error * user_factors[u] - beta * item_factors[i])

        return global_bias, user_bias, item_bias, user_factors, item_factors

    def predict(self, users, movies):
        """
        Make predictions for the given user-movie pairs.

        Parameters:
        users (array-like): User indices for predictions.
        movies (array-like): Movie indices for predictions.

        Returns:
        array-like: Predicted ratings.
        """
        return self._predict_jit(users, movies, self.global_bias, self.user_bias, self.item_bias, self.user_factors, self.item_factors)

    @staticmethod
    @njit
    def _predict_jit(users, movies, global_bias, user_bias, item_bias, user_factors, item_factors):
        """
        Make predictions using Numba's JIT compilation for optimization.

        Parameters:
        users (array-like): User indices for predictions.
        movies (array-like): Movie indices for predictions.
        global_bias (float): Global bias term.
        user_bias (array-like): Bias term for each user.
        item_bias (array-like): Bias term for each item.
        user_factors (array-like): Latent factors for users.
        item_factors (array-like): Latent factors for items.

        Returns:
        array-like: Predicted ratings.
        """
        predictions = np.zeros(len(users))
        for i in range(len(users)):
            u = users[i]  # User index
            m = movies[i]  # Movie index
            predictions[i] = global_bias + user_bias[u] + item_bias[m] + np.dot(user_factors[u], item_factors[m])
        return predictions

# Define the function to compute RMSE
def get_rmse(predictions, targets):
    return np.sqrt(((predictions - targets) ** 2).mean())

# Define the clipping range of ratings
MIN_RATING = 1.0
MAX_RATING = 5.0

# Train the BMF model with a fixed random seed
random_seed = 42
bmf_model = BMF(n_users=number_of_users, n_items=number_of_movies, n_factors=99, n_iter=111, alpha=0.002, beta=0.051, random_state=random_seed)
bmf_model.fit(train_users, train_movies, train_predictions)

# Predictions from BMF model on test data
bfm_test_predictions = bmf_model.predict(test_users, test_movies)

# **Clip BMF predictions before passing to LightGBM**
bfm_test_predictions_clipped = np.clip(bfm_test_predictions, MIN_RATING, MAX_RATING)

# Calculate RMSE using only BMF model on test set
bmf_only_rmse = get_rmse(bfm_test_predictions_clipped, test_predictions)
print(f'BMF Only RMSE: {bmf_only_rmse}')

# Predictions from BMF model on train data for residual calculation
bfm_train_predictions = bmf_model.predict(train_users, train_movies)

# **Clip BMF train predictions**
bfm_train_predictions_clipped = np.clip(bfm_train_predictions, MIN_RATING, MAX_RATING)

# Calculate residuals on train set
train_residuals = train_predictions - bfm_train_predictions_clipped

# Prepare data for LightGBM
X_train_lgb = np.column_stack((train_users, train_movies))
y_train_lgb = train_residuals

# Create LightGBM dataset
lgb_train_data = lgb.Dataset(X_train_lgb, label=y_train_lgb)

# Set LightGBM parameters
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.25,
    'num_leaves': 6,
    'feature_fraction': 0.9,
    'verbose': -1
}

# Train LightGBM model on residuals
lgb_model = lgb.train(params, lgb_train_data, num_boost_round=100)

# Prepare test data for LightGBM
X_test_lgb = np.column_stack((test_users, test_movies))

# Predict residuals on test set
lgb_residuals = lgb_model.predict(X_test_lgb)

# Combine BFM predictions and LightGBM residuals on test set
combined_test_predictions = bfm_test_predictions_clipped + lgb_residuals

# **Clip final combined predictions**
combined_test_predictions_clipped = np.clip(combined_test_predictions, MIN_RATING, MAX_RATING)

# Evaluate the final model
final_rmse = get_rmse(combined_test_predictions_clipped, test_predictions)
print(f'Final RMSE: {final_rmse}')



In [ ]:
# Save predictions to .npz files
predictions_path = 'predictions'
if not os.path.exists(predictions_path):
    os.mkdir(predictions_path)

# Load sample submission file
sample_submission = pd.read_csv('sampleSubmission.csv')

# Extract user and item IDs from the 'Id' column
sample_submission['user_id'] = sample_submission['Id'].apply(lambda x: int(x.split('_')[0][1:]) - 1)
sample_submission['item_id'] = sample_submission['Id'].apply(lambda x: int(x.split('_')[1][1:]) - 1)

# Predict the ratings using the trained model
user_ids = sample_submission['user_id'].values
item_ids = sample_submission['item_id'].values

# Predict using the BMF model
bmf_submission_predictions = bmf_model.predict(user_ids, item_ids)

# **Clip BMF submission predictions**
bmf_submission_predictions_clipped = np.clip(bmf_submission_predictions, MIN_RATING, MAX_RATING)

# Prepare data for LightGBM
X_lgb = np.column_stack((user_ids, item_ids))

# Predict residuals using LightGBM model
lgb_residuals = lgb_model.predict(X_lgb)

# Combine BMF predictions and LightGBM residuals
submission_predictions = bmf_submission_predictions_clipped + lgb_residuals

# **Clip final submission predictions**
submission_predictions_clipped = np.clip(submission_predictions, MIN_RATING, MAX_RATING)

# Save both test and submission predictions in a single .npz file
np.savez_compressed(
    os.path.join(predictions_path, 'bmf_predictions.npz'),
    test=combined_test_predictions_clipped,
    challenge=submission_predictions_clipped
)

# Assign predictions to the sample submission file
sample_submission['Prediction'] = submission_predictions_clipped

csv_path = 'csvs'
if not os.path.exists(csv_path):
    os.mkdir(csv_path)

sample_submission[['Id', 'Prediction']].to_csv(os.path.join(csv_path, 'submission_bmf.csv'), index=False)
